# V3 Baseline PM₂.₅ Regression Models — Delhi NCR

This notebook is a **predictive baseline**, not a causal analysis. It trains Linear Regression, Random Forest Regressor, and LightGBM Regressor on the frozen 2025-context V3 master dataset and the canonical V3 train/test split. The notebook creates separate derived artifacts only under `data/modeling_changes/baseline_predictive_v1/`.

The continuous target is `pm25`. The primary metrics are **R², RMSE, and MAE**; median absolute error is supplementary. Classification metrics such as accuracy, precision, and recall are not reported because they are not primary metrics for a continuous PM₂.₅ regression target. A later alert-classification study would require a separately pre-specified and scientifically justified threshold.

The canonical split is an 80:20 year-balanced station-month holdout. It is **not a spatially independent generalization test** because most stations occur in both train and test. Training-only cross-validation uses five station-grouped folds as a more conservative diagnostic and is not used to retune the test result.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Resolve repository root from the current working directory or notebook path.
root = Path.cwd()
while root != root.parent and not (root / 'data' / 'modeling_changes' / 'datasets' / 'master_modeling_dataset_v3.csv').exists():
    root = root.parent
if not (root / 'data' / 'modeling_changes' / 'datasets' / 'master_modeling_dataset_v3.csv').exists():
    raise FileNotFoundError('Could not locate the frozen V3 repository inputs from the current working directory.')

MASTER_PATH = root / 'data' / 'modeling_changes' / 'datasets' / 'master_modeling_dataset_v3.csv'
TRAIN_PATH = root / 'data' / 'modeling_changes' / 'splits' / 'train.csv'
TEST_PATH = root / 'data' / 'modeling_changes' / 'splits' / 'test.csv'
OUT_ROOT = root / 'data' / 'modeling_changes' / 'baseline_predictive_v1'
RESULTS_DIR = OUT_ROOT / 'results'
PLOTS_DIR = RESULTS_DIR / 'plots'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Repository root:', root)
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', __import__('sklearn').__version__)
print('lightgbm:', __import__('lightgbm').__version__)
print('matplotlib:', __import__('matplotlib').__version__)

Repository root: /home/ubuntu/PM2.5_ACM_Research_Work
Python: 3.12.3
pandas: 3.0.5
numpy: 2.5.1
scikit-learn: 1.9.0
lightgbm: 4.7.0
matplotlib: 3.11.1


## 1. Immutable-input audit

The notebook reads the V3 master, training split, and locked test split. It fails loudly on row-count, schema, key-overlap, duplicate-key, target, finite-value, or station-coverage violations. No input file is rewritten, imputed, scaled, or augmented.

In [2]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def key_frame(df):
    return df[['station', 'year', 'month']].astype({'station': str, 'year': int, 'month': int})

def key_set(df):
    return set(map(tuple, key_frame(df).itertuples(index=False, name=None)))

master = pd.read_csv(MASTER_PATH)
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

required = {'station', 'year', 'month', 'latitude', 'longitude', 'pm25'}
for name, df in [('master', master), ('train', train), ('test', test)]:
    missing = sorted(required - set(df.columns))
    if missing:
        raise AssertionError(f'{name} missing required columns: {missing}')
    if df[['year', 'month']].isna().any().any():
        raise AssertionError(f'{name} has missing temporal keys')
    if not np.isfinite(pd.to_numeric(df['pm25'], errors='coerce')).all():
        raise AssertionError(f'{name} has missing/non-finite pm25')

if len(master) != 1615 or len(train) != 1292 or len(test) != 323:
    raise AssertionError(f'Unexpected row counts: master={len(master)}, train={len(train)}, test={len(test)}')
if list(train.columns) != list(test.columns):
    raise AssertionError('Train and test schemas are not identical and ordered consistently.')
master_extra = sorted(set(train.columns) - set(master.columns))
if master_extra != ['season'] or set(master.columns) != set(train.columns) - set(master_extra):
    raise AssertionError(f'Unexpected split-only columns or schema mismatch: {master_extra}')
if list(train.columns[:len(master.columns)]) != list(master.columns):
    raise AssertionError('The split columns do not preserve the master schema order.')
if master.duplicated(['station', 'year', 'month']).any():
    raise AssertionError('Master contains duplicate station-year-month keys.')
if train.duplicated(['station', 'year', 'month']).any() or test.duplicated(['station', 'year', 'month']).any():
    raise AssertionError('Train or test contains duplicate station-year-month keys.')
train_keys, test_keys, master_keys = key_set(train), key_set(test), key_set(master)
if train_keys & test_keys:
    raise AssertionError('Train/test station-year-month overlap detected.')
if train_keys | test_keys != master_keys:
    raise AssertionError('Train/test key union does not exactly equal the master key universe.')
if sorted(train['year'].unique().tolist()) != [2022, 2023, 2024, 2025] or sorted(test['year'].unique().tolist()) != [2022, 2023, 2024, 2025]:
    raise AssertionError('Both train and test must contain all four V3 years.')
if 'IIT_Delhi' in set(test['station']):
    raise AssertionError('IIT_Delhi must remain train-only under the canonical split.')

numeric_master = master.select_dtypes(include=[np.number]).columns
nonfinite = ~np.isfinite(master[numeric_master].to_numpy(dtype=float))
if nonfinite.any():
    bad = master[numeric_master].columns[np.any(nonfinite, axis=0)].tolist()
    raise AssertionError(f'Non-finite numeric values in V3 master: {bad[:10]}')

input_hashes = {str(p.relative_to(root)): sha256(p) for p in [MASTER_PATH, TRAIN_PATH, TEST_PATH]}
(OUT_ROOT / 'input_hashes.json').write_text(json.dumps(input_hashes, indent=2) + chr(10))
print(json.dumps({
    'master_rows': len(master), 'train_rows': len(train), 'test_rows': len(test),
    'master_stations': int(master['station'].nunique()), 'train_stations': int(train['station'].nunique()),
    'test_stations': int(test['station'].nunique()), 'protected_input_hashes': input_hashes
}, indent=2))

{
  "master_rows": 1615,
  "train_rows": 1292,
  "test_rows": 323,
  "master_stations": 35,
  "train_stations": 35,
  "test_stations": 34,
  "protected_input_hashes": {
    "data/modeling_changes/datasets/master_modeling_dataset_v3.csv": "0204a570682d13a957b58c2622aa626e777cef0731787e594e768a25daa80d84",
    "data/modeling_changes/splits/train.csv": "27feff3f3ebd9c6110a3981c94ce717edb7e0c697ef8593a6a4b277a90012483",
    "data/modeling_changes/splits/test.csv": "04e8562e436b5bfad925e5c564ff1f1b7be6e18498ae1b918e45a108509eaea7"
  }
}


## 2. Feature and target separation

The target is `pm25`. The station identifier, row identity, split-membership columns, future/prediction columns, and target-derived columns are excluded. Numeric environmental, spatial, and temporal predictors are retained, including latitude/longitude as descriptive spatial context. These features support prediction only; they are not causal controls and feature importance is not a causal effect.

In [3]:
TARGET = 'pm25'
EXCLUDED_EXACT = {'station', TARGET}
EXCLUDED_NAME_PARTS = ('prediction', 'predicted', 'residual', 'split_membership', 'fold', 'target_derived')

candidate_numeric = [c for c in train.select_dtypes(include=[np.number]).columns if c not in EXCLUDED_EXACT]
feature_cols = [c for c in candidate_numeric if not any(part in c.lower() for part in EXCLUDED_NAME_PARTS)]
if not feature_cols:
    raise AssertionError('No permissible numeric predictors remain after exclusions.')

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()
y_train = train[TARGET].astype(float).copy()
y_test = test[TARGET].astype(float).copy()
groups_train = train['station'].astype(str)

missing_train = int(X_train.isna().sum().sum())
missing_test = int(X_test.isna().sum().sum())
print('Predictor count:', len(feature_cols))
print('Excluded non-feature columns:', sorted(set(train.columns) - set(feature_cols) - {TARGET}))
print('Missing predictor cells handled inside pipelines: train=', missing_train, 'test=', missing_test)
print('Primary treatment retained as predictive feature:', 'sentinel2_ndvi_mean_1000m' in feature_cols)

Predictor count: 194
Excluded non-feature columns: ['season', 'station']
Missing predictor cells handled inside pipelines: train= 0 test= 0
Primary treatment retained as predictive feature: True


## 3. Reproducible models and preprocessing

Linear Regression uses train-fitted median imputation and standardization. Random Forest and LightGBM use train-fitted median imputation without unnecessary scaling. The tree configurations are transparent baselines rather than large hyperparameter searches. LightGBM is included only after verifying that the package is available.

In [4]:
def make_pipeline(model, scale=False):
    steps = [('imputer', SimpleImputer(strategy='median', keep_empty_features=True))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)

models = {
    'Linear Regression': make_pipeline(LinearRegression(), scale=True),
    'Random Forest': make_pipeline(RandomForestRegressor(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1,
        max_features=1.0, min_samples_leaf=1
    )),
    'LightGBM': make_pipeline(LGBMRegressor(
        n_estimators=400, learning_rate=0.05, num_leaves=31,
        max_depth=-1, min_child_samples=20, subsample=0.9,
        colsample_bytree=0.9, reg_lambda=1.0, random_state=RANDOM_STATE,
        n_jobs=-1, verbosity=-1
    ))
}
print('Models:', ', '.join(models))

Models: Linear Regression, Random Forest, LightGBM


## 4. Training-only station-grouped cross-validation

Five-fold cross-validation is performed on the training set only. Station groups never straddle a fold, reducing the optimistic effect of repeated observations from the same monitor. This remains a training diagnostic, not a replacement for the locked test result or a guarantee of spatial independence.

In [5]:
def rmse(y, pred):
    return float(np.sqrt(mean_squared_error(y, pred)))

def metric_dict(y, pred):
    return {
        'R2': float(r2_score(y, pred)),
        'RMSE': rmse(y, pred),
        'MAE': float(mean_absolute_error(y, pred)),
        'MedianAE': float(median_absolute_error(y, pred)),
    }

cv_rows = []
gkf = GroupKFold(n_splits=5)
for model_name, estimator in models.items():
    fold_rows = []
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups_train), start=1):
        estimator.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        pred = estimator.predict(X_train.iloc[va_idx])
        m = metric_dict(y_train.iloc[va_idx], pred)
        m.update({'model': model_name, 'fold': fold, 'n_train': len(tr_idx), 'n_validation': len(va_idx), 'stations_validation': groups_train.iloc[va_idx].nunique()})
        fold_rows.append(m)
        cv_rows.append(m)
    print(model_name, 'CV RMSE mean:', round(np.mean([r['RMSE'] for r in fold_rows]), 4))

cv_folds = pd.DataFrame(cv_rows)
cv_summary = cv_folds.groupby('model').agg(
    CV_R2_mean=('R2', 'mean'), CV_R2_std=('R2', 'std'),
    CV_RMSE_mean=('RMSE', 'mean'), CV_RMSE_std=('RMSE', 'std'),
    CV_MAE_mean=('MAE', 'mean'), CV_MAE_std=('MAE', 'std')
).reset_index()

Linear Regression CV RMSE mean: 282.4765


Random Forest CV RMSE mean: 25.0756


LightGBM CV RMSE mean: 24.826


## 5. Final fit and locked-test evaluation

Each model is now fit once on all training rows. The test split is used only for final diagnostic evaluation; it is not used for preprocessing, feature selection, tuning, or model choice during the notebook run.

In [6]:
fit_models = {}
predictions = pd.DataFrame({
    'station': test['station'].astype(str).values,
    'year': test['year'].astype(int).values,
    'month': test['month'].astype(int).values,
    'latitude': test['latitude'].astype(float).values,
    'longitude': test['longitude'].astype(float).values,
    'observed_pm25': y_test.values,
})
metric_rows = []
for model_name, estimator in models.items():
    estimator.fit(X_train, y_train)
    fit_models[model_name] = estimator
    pred_train = estimator.predict(X_train)
    pred_test = estimator.predict(X_test)
    predictions[f'{model_name}_predicted_pm25'] = pred_test
    train_m = metric_dict(y_train, pred_train)
    test_m = metric_dict(y_test, pred_test)
    row = {'model': model_name}
    row.update({f'train_{k}': v for k, v in train_m.items()})
    row.update({f'test_{k}': v for k, v in test_m.items()})
    metric_rows.append(row)

metrics = pd.DataFrame(metric_rows).merge(cv_summary, on='model', how='left')
metrics = metrics[['model', 'train_R2', 'train_RMSE', 'train_MAE', 'test_R2', 'test_RMSE', 'test_MAE', 'test_MedianAE',
                   'CV_R2_mean', 'CV_R2_std', 'CV_RMSE_mean', 'CV_RMSE_std', 'CV_MAE_mean', 'CV_MAE_std']]
print(metrics.to_string(index=False, float_format=lambda x: f'{x:.6f}'))

            model  train_R2  train_RMSE  train_MAE  test_R2  test_RMSE  test_MAE  test_MedianAE  CV_R2_mean  CV_R2_std  CV_RMSE_mean  CV_RMSE_std  CV_MAE_mean  CV_MAE_std
Linear Regression  0.875000   23.684128  18.146379 0.820184  29.096270 21.637563      17.619091  -32.901854  56.818620    282.476452   302.729850    91.106494   52.806089
    Random Forest  0.990239    6.618224   4.295660 0.908961  20.703122 11.390408       6.497000    0.855610   0.049957     25.075591     4.411473    16.526640    2.659745
         LightGBM  0.999619    1.307879   0.628834 0.924285  18.880479 10.088022       6.682582    0.858258   0.052242     24.826004     4.879498    16.790590    2.824722


## 6. Year-wise, season-wise, and residual diagnostics

Year and season summaries are descriptive stability checks. Extreme pollution observations are retained. RMSE is compared with MAE and with an extreme-event subgroup defined using the **training-only 95th percentile**; no row is deleted.

In [7]:
def season_from_month(month):
    if month in (12, 1, 2): return 'Winter'
    if month in (3, 4, 5, 6): return 'Summer'
    if month in (7, 8, 9): return 'Monsoon'
    return 'Post-monsoon'

test_diag = predictions.copy()
test_diag['season'] = test_diag['month'].map(season_from_month)
train_p95 = float(y_train.quantile(0.95))
test_diag['extreme_train_p95'] = test_diag['observed_pm25'] >= train_p95

def grouped_metrics(pred_col, group_col):
    rows = []
    for group, frame in test_diag.groupby(group_col, sort=True):
        y = frame['observed_pm25']
        p = frame[pred_col]
        m = metric_dict(y, p)
        m.update({'group': group, 'n': len(frame)})
        rows.append(m)
    return pd.DataFrame(rows)

year_rows, season_rows = [], []
for model_name in models:
    pred_col = f'{model_name}_predicted_pm25'
    ym = grouped_metrics(pred_col, 'year'); ym.insert(0, 'model', model_name); year_rows.append(ym)
    sm = grouped_metrics(pred_col, 'season'); sm.insert(0, 'model', model_name); season_rows.append(sm)
yearly_metrics = pd.concat(year_rows, ignore_index=True)
seasonal_metrics = pd.concat(season_rows, ignore_index=True)

residual_rows = []
for model_name in models:
    pred_col = f'{model_name}_predicted_pm25'
    residual = test_diag['observed_pm25'] - test_diag[pred_col]
    abs_resid = residual.abs()
    high = test_diag['extreme_train_p95']
    high_rmse = rmse(test_diag.loc[high, 'observed_pm25'], test_diag.loc[high, pred_col]) if high.any() else np.nan
    nonhigh_rmse = rmse(test_diag.loc[~high, 'observed_pm25'], test_diag.loc[~high, pred_col]) if (~high).any() else np.nan
    residual_rows.append({
        'model': model_name, 'mean_residual_observed_minus_predicted': float(residual.mean()),
        'median_residual': float(residual.median()), 'residual_std': float(residual.std(ddof=1)),
        'worst_positive_residual_underprediction': float(residual.max()),
        'worst_negative_residual_overprediction': float(residual.min()),
        'mean_abs_residual': float(abs_resid.mean()), 'high_pm25_rmse_train_p95_group': high_rmse,
        'non_high_pm25_rmse': nonhigh_rmse,
        'high_to_non_high_rmse_ratio': float(high_rmse / nonhigh_rmse) if nonhigh_rmse and np.isfinite(nonhigh_rmse) else np.nan,
        'abs_residual_observed_corr': float(np.corrcoef(abs_resid, test_diag['observed_pm25'])[0, 1])
    })
residual_summary = pd.DataFrame(residual_rows)

# Station-level error summary for the spatial map and anomaly report.
best_model = metrics.sort_values(['test_RMSE', 'test_MAE', 'model']).iloc[0]['model']
best_pred_col = f'{best_model}_predicted_pm25'
test_diag['best_model_abs_error'] = (test_diag['observed_pm25'] - test_diag[best_pred_col]).abs()
station_errors = test_diag.groupby(['station', 'latitude', 'longitude'], as_index=False).agg(
    observed_pm25_mean=('observed_pm25', 'mean'), mean_abs_error=('best_model_abs_error', 'mean'),
    n_test=('observed_pm25', 'size')
)

print('Best test model by RMSE:', best_model)
print('Training-only 95th percentile PM2.5:', round(train_p95, 4))
print('Target distribution:', json.dumps({
    'min': float(master[TARGET].min()), 'max': float(master[TARGET].max()),
    'median': float(master[TARGET].median()), 'p95': float(master[TARGET].quantile(.95)),
    'p99': float(master[TARGET].quantile(.99))
}, indent=2))

Best test model by RMSE: LightGBM
Training-only 95th percentile PM2.5: 239.1335
Target distribution: {
  "min": 5.0,
  "max": 313.15,
  "median": 81.65,
  "p95": 239.406,
  "p99": 286.51159999999993
}


## 7. Six research-quality static visualizations

Exactly six main PNG figures are produced. They are descriptive and predictive. None is interpreted as causal proof, and no color scale or axis is selected to force an attractive conclusion.

In [8]:
sns.set_theme(style='whitegrid', context='notebook', font_scale=1.0)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'axes.titleweight': 'bold'})
colors = {'Linear Regression': '#3b5b92', 'Random Forest': '#2a9d8f', 'LightGBM': '#e76f51'}
model_order = list(models)

# 01: three separate panels avoid mixing incompatible units.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, metric_name, label in zip(axes, ['test_R2', 'test_RMSE', 'test_MAE'], ['R² (higher is better)', 'RMSE (µg/m³)', 'MAE (µg/m³)']):
    vals = [float(metrics.loc[metrics.model == m, metric_name].iloc[0]) for m in model_order]
    ax.bar(model_order, vals, color=[colors[m] for m in model_order])
    ax.set_title(label); ax.tick_params(axis='x', rotation=35); ax.set_ylabel(label)
    for i, v in enumerate(vals): ax.text(i, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9)
fig.suptitle('V3 baseline model performance on locked test split', y=1.03)
fig.tight_layout(); fig.savefig(PLOTS_DIR / '01_model_performance.png', bbox_inches='tight'); plt.close(fig)

# 02: observed vs predicted.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
lims = [float(min(test_diag.observed_pm25.min(), test_diag[[f'{m}_predicted_pm25' for m in model_order]].min().min())),
        float(max(test_diag.observed_pm25.max(), test_diag[[f'{m}_predicted_pm25' for m in model_order]].max().max()))]
for ax, m in zip(axes, model_order):
    pc = f'{m}_predicted_pm25'; ax.scatter(test_diag.observed_pm25, test_diag[pc], s=22, alpha=.65, color=colors[m], edgecolor='none')
    ax.plot(lims, lims, '--', color='black', linewidth=1)
    row = metrics.loc[metrics.model == m].iloc[0]
    ax.set_title(m); ax.set_xlabel('Observed PM₂.₅ (µg/m³)'); ax.set_ylabel('Predicted PM₂.₅ (µg/m³)')
    ax.text(.04, .96, f"R²={row.test_R2:.3f}" + chr(10) + f"RMSE={row.test_RMSE:.2f}", transform=ax.transAxes, va='top', bbox=dict(facecolor='white', alpha=.8, edgecolor='none'))
    ax.set_xlim(lims); ax.set_ylim(lims)
fig.suptitle('Observed versus predicted PM₂.₅ — descriptive prediction check', y=1.03)
fig.tight_layout(); fig.savefig(PLOTS_DIR / '02_observed_vs_predicted.png', bbox_inches='tight'); plt.close(fig)

# 03: residual diagnostics.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, m in zip(axes, model_order):
    pc = f'{m}_predicted_pm25'; resid = test_diag.observed_pm25 - test_diag[pc]
    ax.scatter(test_diag[pc], resid, s=22, alpha=.65, color=colors[m], edgecolor='none')
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title(m); ax.set_xlabel('Predicted PM₂.₅ (µg/m³)'); ax.set_ylabel('Observed − predicted (µg/m³)')
fig.suptitle('Residual diagnostics on the locked test split', y=1.03)
fig.tight_layout(); fig.savefig(PLOTS_DIR / '03_residual_diagnostics.png', bbox_inches='tight'); plt.close(fig)

# 04: station error map. It is a coordinate map, not a geographic causal surface.
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(station_errors.longitude, station_errors.latitude, c=station_errors.mean_abs_error, s=55 + 12 * np.sqrt(station_errors.n_test), cmap='magma', edgecolor='black', linewidth=.35)
for _, r in station_errors.iterrows():
    if r.mean_abs_error >= station_errors.mean_abs_error.quantile(.9):
        ax.annotate(str(r.station), (r.longitude, r.latitude), xytext=(4, 4), textcoords='offset points', fontsize=7)
ax.set_title(f'Station-level mean absolute test error — {best_model}')
ax.set_xlabel('Longitude (degrees)'); ax.set_ylabel('Latitude (degrees)')
cb = fig.colorbar(sc, ax=ax); cb.set_label('Mean absolute error (µg/m³)')
ax.text(.02, .02, 'Descriptive station map; not causal proof', transform=ax.transAxes, fontsize=8, bbox=dict(facecolor='white', alpha=.8, edgecolor='none'))
fig.tight_layout(); fig.savefig(PLOTS_DIR / '04_spatial_error_map.png', bbox_inches='tight'); plt.close(fig)

# 05: predictive feature importance for RF and LightGBM.
def feature_group(name):
    n = name.lower()
    if any(x in n for x in ['ndvi', 'evi', 'ndwi', 'vegetation']): return 'Vegetation'
    if 'no2' in n: return 'NO₂'
    if any(x in n for x in ['era5', 'rh_', 'wind', 'blh', 'temperature']): return 'Meteorology'
    if any(x in n for x in ['lst', 'land_surface']): return 'Thermal'
    if any(x in n for x in ['population', 'worldpop', 'ghsl']): return 'Population'
    if 'road' in n: return 'Roads'
    if any(x in n for x in ['dynamicworld', 'worldcover', 'landcover', 'built_frac', 'bare_frac', 'water_frac']): return 'Land cover'
    if n in {'latitude', 'longitude'}: return 'Spatial'
    if any(x in n for x in ['year', 'month', 'season', 'sin', 'cos']): return 'Temporal'
    return 'Other'

importance_frames = []
for m in ['Random Forest', 'LightGBM']:
    fitted_model = fit_models[m].named_steps['model']
    if m == 'LightGBM':
        vals = fitted_model.booster_.feature_importance(importance_type='gain')
        measure = 'gain'
    else:
        vals = fitted_model.feature_importances_
        measure = 'split_impurity'
    imp = pd.DataFrame({'model': m, 'feature': feature_cols, 'importance': vals, 'importance_measure': measure})
    imp['feature_group'] = imp['feature'].map(feature_group)
    imp['importance_share'] = imp['importance'] / imp['importance'].sum() if imp['importance'].sum() else 0
    importance_frames.append(imp)
feature_importance = pd.concat(importance_frames, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, m in zip(axes, ['Random Forest', 'LightGBM']):
    top = feature_importance[feature_importance.model == m].nlargest(15, 'importance').sort_values('importance')
    pal = {'Vegetation':'#6a994e','NO₂':'#8338ec','Meteorology':'#457b9d','Thermal':'#f4a261','Population':'#e76f51','Roads':'#8d99ae','Land cover':'#90be6d','Spatial':'#264653','Temporal':'#bc6c25','Other':'#999999'}
    ax.barh(top.feature, top.importance, color=[pal.get(x, '#999999') for x in top.feature_group])
    ax.set_title(f'{m} importance ({top.importance_measure.iloc[0]})'); ax.set_xlabel('Importance; not a causal effect')
fig.suptitle('Top predictive features in tree-based baselines', y=1.01)
fig.tight_layout(); fig.savefig(PLOTS_DIR / '05_feature_importance.png', bbox_inches='tight'); plt.close(fig)

# 06: pre-specified primary green-cover relationship, descriptive only.
ndvi_col = 'sentinel2_ndvi_mean_1000m'
if ndvi_col not in master.columns:
    raise AssertionError(f'Expected primary green-cover feature missing: {ndvi_col}')
plot_df = test[['year', ndvi_col, TARGET]].copy().dropna()
fig, ax = plt.subplots(figsize=(8, 5.5))
sc = ax.scatter(plot_df[ndvi_col], plot_df[TARGET], c=plot_df['year'], cmap='viridis', s=24, alpha=.7, edgecolor='none')
if len(plot_df) >= 2 and plot_df[ndvi_col].nunique() > 1:
    coef = np.polyfit(plot_df[ndvi_col], plot_df[TARGET], 1)
    xx = np.linspace(plot_df[ndvi_col].min(), plot_df[ndvi_col].max(), 100)
    ax.plot(xx, coef[0] * xx + coef[1], color='black', linestyle='--', linewidth=1.5, label='Descriptive linear trend')
ax.set_title('Observed PM₂.₅ versus primary Sentinel-2 NDVI (test split)')
ax.set_xlabel('Sentinel-2 NDVI mean, 1,000 m (unitless)'); ax.set_ylabel('Observed PM₂.₅ (µg/m³)')
cb = fig.colorbar(sc, ax=ax); cb.set_label('Year')
ax.legend(loc='best'); ax.text(.02, .02, 'Descriptive association; not causal evidence', transform=ax.transAxes, fontsize=8, bbox=dict(facecolor='white', alpha=.8, edgecolor='none'))
fig.tight_layout(); fig.savefig(PLOTS_DIR / '06_environmental_relationship.png', bbox_inches='tight'); plt.close(fig)

print('Saved exactly six main plots:', sorted(p.name for p in PLOTS_DIR.glob('*.png')))

Saved exactly six main plots: ['01_model_performance.png', '02_observed_vs_predicted.png', '03_residual_diagnostics.png', '04_spatial_error_map.png', '05_feature_importance.png', '06_environmental_relationship.png']


## 8. Automated findings and final artifact export

Findings use cautious language such as “predictively associated,” “appears influential,” and “requires further investigation.” The notebook does not call any result statistically significant and does not convert predictive importance into a causal effect.

In [9]:
# Automated diagnostics.
findings = []

def add(flag, text):
    if flag:
        findings.append(('FLAG', text))

best_row = metrics.sort_values(['test_RMSE', 'test_MAE', 'model']).iloc[0]
linear_row = metrics.loc[metrics.model == 'Linear Regression'].iloc[0]
for m in ['Random Forest', 'LightGBM']:
    row = metrics.loc[metrics.model == m].iloc[0]
    add(row.test_RMSE < linear_row.test_RMSE * 0.9, f'{m} has at least 10% lower test RMSE than Linear Regression: predictive nonlinear improvement should be investigated.')
add(metrics['train_R2'].max() - metrics['test_R2'].max() > 0.15, 'Train-to-test R² gap exceeds 0.15 for at least one model; potential overfitting requires investigation.')
for _, row in residual_summary.iterrows():
    add(row.high_to_non_high_rmse_ratio > 1.25, f"{row.model} has substantially larger RMSE for observations above the training 95th percentile; extreme pollution events influence error.")
    add(abs(row.mean_residual_observed_minus_predicted) > 5, f"{row.model} shows a mean residual magnitude above 5 µg/m³; systematic bias should be investigated.")
    add(row.abs_residual_observed_corr > 0.3, f"{row.model} absolute residuals rise with observed PM₂.₅; possible heteroscedasticity/high-pollution difficulty.")

station_cut = station_errors.mean_abs_error.median() + 1.5 * (station_errors.mean_abs_error.quantile(.75) - station_errors.mean_abs_error.quantile(.25))
bad_stations = station_errors.loc[station_errors.mean_abs_error > station_cut, 'station'].astype(str).tolist()
add(bool(bad_stations), f'Unusually high station-level error under {best_model}: {bad_stations}. Spatial station behavior is descriptive and requires further investigation.')
for group_name, frame in [('year', yearly_metrics), ('season', seasonal_metrics)]:
    for m in model_order:
        sub = frame[frame.model == m]
        if len(sub) >= 3:
            cut = sub.RMSE.median() + 1.5 * (sub.RMSE.quantile(.75) - sub.RMSE.quantile(.25))
            unusual = sub.loc[sub.RMSE > cut, 'group'].astype(str).tolist()
            add(bool(unusual), f'{m} has unusually high {group_name}-wise RMSE in {unusual}; temporal regime differences require investigation.')

# Importance-group summaries and multicollinearity audit on training predictors only.
importance_group = feature_importance.groupby(['model', 'feature_group'], as_index=False)['importance_share'].sum().sort_values(['model', 'importance_share'], ascending=[True, False])
for m in ['Random Forest', 'LightGBM']:
    top_group = importance_group[importance_group.model == m].iloc[0]
    add(True, f"{m}'s largest aggregate predictive importance group is {top_group.feature_group} ({top_group.importance_share:.1%}); this is not causal evidence.")

corr = X_train.corr(numeric_only=True).abs()
pairs = []
for i, col_a in enumerate(corr.columns):
    for col_b in corr.columns[i+1:]:
        if corr.loc[col_a, col_b] >= 0.90:
            pairs.append((col_a, col_b, float(corr.loc[col_a, col_b])))
add(bool(pairs), f'High training-feature collinearity (absolute correlation >= 0.90) found in {len(pairs)} pairs; tree importance and linear coefficients should not be interpreted independently.')

findings_lines = [
    'V3 BASELINE PREDICTIVE MODELING FINDINGS',
    '=========================================',
    f'Best locked-test model by RMSE: {best_model}',
    'The analysis is predictive and observational; no baseline result establishes causality.',
    'Primary metrics are R2, RMSE, and MAE. Accuracy, precision, and recall are intentionally not reported for continuous PM2.5 regression.',
    '',
    'Model metrics:',
    metrics.to_string(index=False, float_format=lambda x: f'{x:.6f}'),
    '',
    'Residual summaries:',
    residual_summary.to_string(index=False, float_format=lambda x: f'{x:.6f}'),
    '',
    'Automated flags and notes:'
]
findings_lines.extend(f'[{kind}] {text}' for kind, text in findings)
findings_lines.extend([
    '',
    'Interpretation guardrails:',
    '- The year-balanced split is not spatially independent because most stations occur in both train and test.',
    '- Feature importance is predictive/exploratory, not a causal effect or treatment ranking.',
    '- Extreme PM2.5 observations are retained; elevated RMSE for them is a diagnostic, not a reason to remove them.',
    '- The primary NDVI relationship figure is descriptive and cannot establish that vegetation reduces PM2.5.',
    '- DML estimates and dependence-aware intervals remain the separate causal-inference workstream.',
])
findings_text = chr(10).join(findings_lines) + chr(10)

# Persist only new derived artifacts under the new baseline folder.
metrics.to_csv(RESULTS_DIR / 'baseline_model_metrics.csv', index=False)
yearly_metrics.to_csv(RESULTS_DIR / 'yearly_model_metrics.csv', index=False)
seasonal_metrics.to_csv(RESULTS_DIR / 'seasonal_model_metrics.csv', index=False)
residual_summary.to_csv(RESULTS_DIR / 'residual_summary.csv', index=False)
feature_importance.to_csv(RESULTS_DIR / 'feature_importance.csv', index=False)
cv_folds.to_csv(RESULTS_DIR / 'cross_validation_folds.csv', index=False)
predictions.to_csv(RESULTS_DIR / 'test_predictions.csv', index=False)
station_errors.to_csv(RESULTS_DIR / 'station_error_summary.csv', index=False)
importance_group.to_csv(RESULTS_DIR / 'feature_group_importance.csv', index=False)
(RESULTS_DIR / 'findings_report.txt').write_text(findings_text)
(OUT_ROOT / 'run_config.json').write_text(json.dumps({
    'random_state': RANDOM_STATE, 'master_input': str(MASTER_PATH.relative_to(root)),
    'train_input': str(TRAIN_PATH.relative_to(root)), 'test_input': str(TEST_PATH.relative_to(root)),
    'feature_count': len(feature_cols), 'models': list(models),
    'cv': '5-fold GroupKFold grouped by station on training only',
    'target': TARGET, 'best_test_model_by_rmse': best_model,
    'protected_paths': ['data/modeling_changes/datasets/', 'data/modeling_changes/splits/']
}, indent=2) + chr(10))

print(findings_text)
print('Exported results to', RESULTS_DIR)

V3 BASELINE PREDICTIVE MODELING FINDINGS
Best locked-test model by RMSE: LightGBM
The analysis is predictive and observational; no baseline result establishes causality.
Primary metrics are R2, RMSE, and MAE. Accuracy, precision, and recall are intentionally not reported for continuous PM2.5 regression.

Model metrics:
            model  train_R2  train_RMSE  train_MAE  test_R2  test_RMSE  test_MAE  test_MedianAE  CV_R2_mean  CV_R2_std  CV_RMSE_mean  CV_RMSE_std  CV_MAE_mean  CV_MAE_std
Linear Regression  0.875000   23.684128  18.146379 0.820184  29.096270 21.637563      17.619091  -32.901854  56.818620    282.476452   302.729850    91.106494   52.806089
    Random Forest  0.990239    6.618224   4.295660 0.908961  20.703122 11.390408       6.497000    0.855610   0.049957     25.075591     4.411473    16.526640    2.659745
         LightGBM  0.999619    1.307879   0.628834 0.924285  18.880479 10.088022       6.682582    0.858258   0.052242     24.826004     4.879498    16.790590    2.82

## Final summary

This notebook establishes a transparent predictive baseline on the frozen V3 context. It should be used to describe prediction quality, temporal/station error patterns, and exploratory feature usefulness. It must not be used to claim that greenery causes PM₂.₅ reductions; that question is addressed separately by the pre-specified DML designs and their dependence-aware uncertainty intervals.